# Phase Gates — Amazon Braket

Phase gates (S, T) rotate the qubit's state around the z-axis of the
Bloch sphere without changing the measurement probabilities in the
computational basis.

In [ ]:
import json

from braket.circuit import Circuit
from braket.devices import LocalSimulator

In [ ]:
device = LocalSimulator()

## Hadamard creates equal superposition

In [ ]:
circuit = Circuit()
circuit.h(0)
result = device.run(circuit, shots=0).result()
print(f"H|0> amplitudes: {result.result_types[0].value}")
print(f"H|0> probabilities: {result.result_types[1].value}")

## S gate: pi/2 phase

$S|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + i|1\rangle)$

In [ ]:
circuit = Circuit()
circuit.h(0)
circuit.s(0)
result = device.run(circuit, shots=0).result()
print(f"S|+> amplitudes: {result.result_types[0].value}")
print(f"S|+> probabilities: {result.result_types[1].value}")
print("S adds pi/2 phase to |1>: same probs, different phase")

## S-dagger reverses S: S\u2020S|+\rangle = |+\rangle

In [ ]:
circuit = Circuit()
circuit.h(0)
circuit.s(0)
circuit.s(0).dagger()
result = device.run(circuit, shots=0).result()
print(f"S+S|+> amplitudes: {result.result_types[0].value}")

## T gate: pi/4 phase

$T|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + e^{i\pi/4}|1\rangle)$

In [ ]:
circuit = Circuit()
circuit.h(0)
circuit.t(0)
result = device.run(circuit, shots=0).result()
print(f"T|+> amplitudes: {result.result_types[0].value}")
print(f"T|+> probabilities: {result.result_types[1].value}")
print("T adds pi/4 phase to |1>: same probs as |+>, finer phase")

## T-dagger reverses T: T\u2020T|+\rangle = |+\rangle

In [ ]:
circuit = Circuit()
circuit.h(0)
circuit.t(0)
circuit.t(0).dagger()
result = device.run(circuit, shots=0).result()
print(f"T+T|+> amplitudes: {result.result_types[0].value}")

## Composed phases: T\u00b7S

S adds $\pi/2$, T adds $\pi/4$, total = $3\pi/4$.

In [ ]:
circuit = Circuit()
circuit.h(0)
circuit.s(0)
circuit.t(0)
result = device.run(circuit, shots=0).result()
print(f"T*S|+> probabilities: {result.result_types[1].value}")
print("Phases compose: S adds pi/2, T adds pi/4, total = 3pi/4")

## Phase gates don't change Z-basis measurement

All phase gates leave the computational-basis measurement at 50/50.

In [ ]:
circuits = {
    "H|0>":   [("h",)],
    "S|+>":   [("h",), ("s",)],
    "T|+>":   [("h",), ("t",)],
    "T*S|+>": [("h",), ("s",), ("t",)],
}

for name, gate_list in circuits.items():
    circuit = Circuit()
    for gates in gate_list:
        for g in gates:
            getattr(circuit, g)(0)
    result = device.run(circuit, shots=2000).result()
    probs = result.result_types[1].value
    print(f"  {name}: {json.dumps({k: round(v, 4) for k, v in probs.items()})}")

print("All ~50/50 — phase is invisible to Z-measurement.")